# SPEC2RTL & PROMPT2SILICON: CĂN CHỈNH MÔ HÌNH BẰNG DPO (DIRECT PREFERENCE OPTIMIZATION)
### Kế hoạch Tuần 2: Căn chỉnh sở thích phần cứng (Hardware Preference Alignment)
**Nhóm 2:** Huấn luyện AI Cục bộ & Tinh chỉnh Nâng cao (Fine-Tuning)
**Mục tiêu:** Huấn luyện mô hình `Qwen2.5-Coder-3B` bằng kỹ thuật **DPO** với 1,098 cặp dữ liệu `(chosen vs rejected)` để triệt tiêu dứt điểm:
1. **Chốt ẩn (Inferred Latches)**: Phạt nặng code thiếu nhánh `default:` hoặc `else`.
2. **Chân lơ lửng (Dangling Pins)**: Phạt nặng code không nối cổng `btn_in` (lỗi debouncer).
3. **Race Condition**: Phạt nặng gán blocking `=` trong khối tuần tự `posedge clk`.
4. **Chi phí**: 0 VNĐ trên GPU T4 Google Colab.

## Bước 1: Kiểm tra phần cứng GPU T4 Google Colab
*Vào menu Runtime -> Change runtime type -> Chọn T4 GPU (miễn phí).* 

In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## Bước 2: Cài đặt Unsloth & Thư viện Huấn luyện DPO (TRL)
Unsloth hỗ trợ tối ưu bộ nhớ gấp 5 lần và tăng tốc huấn luyện DPO nhanh hơn 2x so với thông thường.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets matplotlib
!pip install --upgrade trl

## Bước 3: Nạp Mô hình Nền Qwen2.5-Coder-3B & Cấu hình LoRA
Sử dụng kỹ thuật 4-bit quantization để chạy mượt mà trên 15GB VRAM của T4 GPU.

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer() # Tối ưu hóa bộ nhớ cho DPO
import torch

max_seq_length = 2048
model_name = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"

print(f"[*] Đang nạp mô hình: {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    dtype = None,
)

# Cấu hình LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print(">>> Đã nạp và cấu hình mô hình LoRA thành công!")

## Bước 4: Tự động tải Tập dữ liệu DPO Căn chỉnh Phần cứng (1,098 Cặp)
Tải trực tiếp từ GitHub repository của đồ án: `LamGianggg/TKLLS`.

In [ ]:
import urllib.request
import json
from datasets import Dataset

DPO_URL = "https://raw.githubusercontent.com/LamGianggg/TKLLS/main/training/dataset_dpo_preference.json"
print(f"[*] Đang tải dữ liệu DPO từ: {DPO_URL}...")
urllib.request.urlretrieve(DPO_URL, "dataset_dpo_preference.json")

with open("dataset_dpo_preference.json", "r", encoding="utf-8") as f:
    raw_dpo_data = json.load(f)

print(f">>> Đã nạp thành công {len(raw_dpo_data)} cặp dữ liệu DPO (Chosen vs Rejected)!")

# Chuẩn bị dữ liệu theo format chuẩn của DPOTrainer
formatted_data = {
    "prompt": [],
    "chosen": [],
    "rejected": []
}

system_prompt = "Bạn là chuyên gia thiết kế vi mạch RTL và Verilog synthesizable chuẩn công nghiệp IEEE 1364-2001. Luôn sinh mã chuẩn sạch, đầy đủ cổng, không tạo chốt ẩn (no latches) và đúng quy tắc gán tuần tự/tổ hợp."

for item in raw_dpo_data:
    p_text = f"<|im_start|>system\n{system_prompt}<|im_end|>\n<|im_start|>user\n{item['prompt']}<|im_end|>\n<|im_start|>assistant\n"
    c_text = f"{item['chosen']}<|im_end|>"
    r_text = f"{item['rejected']}<|im_end|>"
    
    formatted_data["prompt"].append(p_text)
    formatted_data["chosen"].append(c_text)
    formatted_data["rejected"].append(r_text)

dpo_dataset = Dataset.from_dict(formatted_data)
print(f">>> Dataset DPO đã sẵn sàng: {len(dpo_dataset)} mẫu!")
print("Ví dụ mẫu 0 (Lý do Rejected):")
print(f" - Lỗi bị phạt: {raw_dpo_data[0].get('rejected_error_type', 'N/A')}")

## Bước 5: Cấu hình và Chạy Huấn luyện DPO (Direct Preference Optimization)
Mô hình học cách tối đa hóa xác suất chọn `chosen` (code chuẩn) và giảm thiểu xác suất chọn `rejected` (code có chốt ẩn, chân lơ lửng, race condition).

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_config = DPOConfig(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,
    warmup_ratio = 0.1,
    num_train_epochs = 1, # 1 Epoch trên 1,098 cặp mất ~15-18 phút trên T4 GPU
    learning_rate = 5e-6,
    fp16 = not is_bfloat16_supported(),
    bf16 = is_bfloat16_supported(),
    logging_steps = 1,
    optim = "adamw_8bit",
    weight_decay = 0.01,
    max_prompt_length = 512,
    max_length = 1024,
    beta = 0.1, # Hệ số phạt DPO (Implicit Reward Margin)
    seed = 3407,
    output_dir = "dpo_outputs",
    report_to = "none",
)

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,
    args = dpo_config,
    beta = 0.1,
    train_dataset = dpo_dataset,
    tokenizer = tokenizer,
    max_length = 1024,
    max_prompt_length = 512,
)

print(">>> BẮT ĐẦU HUẤN LUYỆN DPO (CĂN CHỈNH PHẦN CỨNG)...")
dpo_stats = dpo_trainer.train()
print(">>> HUẤN LUYỆN DPO THÀNH CÔNG RỰC RỠ!")

## Bước 5.5: Vẽ Biểu Đồ Hội Tụ DPO Loss (Lấy hình ảnh cho Slide Báo Cáo)
Đồ thị đường cong mất mát (Loss Curve) dốc xuống chứng minh mô hình đã học thành công cách loại trừ code lỗi.

In [ ]:
import matplotlib.pyplot as plt

# Trích xuất lịch sử Loss từ trainer
history = dpo_trainer.state.log_history
steps = [x['step'] for x in history if 'loss' in x]
losses = [x['loss'] for x in history if 'loss' in x]

if steps and losses:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, losses, marker='o', color='#2563eb', linewidth=2, label='DPO Loss')
    plt.title('Biểu Đồ Hội Tụ DPO Loss (Hardware Preference Alignment Convergence)', fontsize=14, fontweight='bold')
    plt.xlabel('Số bước huấn luyện (Training Steps)', fontsize=12)
    plt.ylabel('Hàm mất mát (DPO Loss)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig('dpo_loss_convergence.png', dpi=300)
    plt.show()
    print('>>> Đã lưu ảnh biểu đồ: dpo_loss_convergence.png (Tải về để dán vào Slide Báo Cáo!)')
else:
    print('Chưa có lịch sử loss để vẽ đồ thị.')

## Bước 6: Kiểm tra Suy luận Sau khi Căn chỉnh DPO
Thử nghiệm bài toán khử rung nút bấm `button_debouncer` để xác nhận cổng `btn_in` đã được kết nối và không bị cô lập.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = """Hãy thiết kế module Verilog synthesizable cho mạch khử rung nút bấm: button_debouncer

[Đặc tả cổng]:
- Inputs: clk (xung nhịp), rst (reset tích cực cao), btn_in (tín hiệu vật lý từ nút bấm cơ học).
- Outputs: btn_clean (tín hiệu nút sạch sau khi khử rung), btn_pressed_pulse (xung 1 chu kỳ khi nhấn).

[Yêu cầu]:
- Tuyệt đối không để chân btn_in bị lơ lửng (dangling pin).
- Lấy mẫu btn_in qua thanh ghi dịch hoặc bộ đếm thời gian ổn định."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer)
print("=== KẾT QUẢ MÔ HÌNH SINH CODE SAU DPO ===")
_ = model.generate(input_ids = inputs, streamer = streamer, max_new_tokens = 512, use_cache = True)

## Bước 7: Xuất và Nén File Trọng số DPO LoRA
Lưu LoRA adapter DPO vào file zip để tải về và tích hợp vào hệ thống Spec2RTL.

In [ ]:
model.save_pretrained("qwen2.5_3b_verilog_dpo")
tokenizer.save_pretrained("qwen2.5_3b_verilog_dpo")
print(">>> Đã lưu DPO LoRA adapter vào thư mục: qwen2.5_3b_verilog_dpo")

# Nén zip
!zip -r qwen2.5_3b_verilog_dpo.zip qwen2.5_3b_verilog_dpo
print(">>> Đã tạo file nén: qwen2.5_3b_verilog_dpo.zip sẵn sàng tải về!")